# From Solo LLM to Agent Swarm
Live 60-minute workshop. Run from repository root with a private vLLM endpoint; figures are measured, not fabricated.

## 1. Load sample and client (12–20 min)
The contradiction is 04:00 arrival followed by 03:30 departure.

In [ ]:
import asyncio, json, time
from pathlib import Path
from agents.workflow import IncidentWorkshop, check_timeline, benchmark
incident = json.loads(Path('data/incidents.json').read_text())[0]
workshop = IncidentWorkshop()
print(json.dumps(incident, indent=2))

In [ ]:
solo_start=time.perf_counter()
solo = await workshop.solo(incident)
print(solo.text)
print('wall seconds:', round(time.perf_counter()-solo_start, 3))

## 2. Specialist agents (20–30 min)
Compare sequential and parallel execution. Same underlying LLM server, independent agent prompts.

In [ ]:
t=time.perf_counter()
sequential=await workshop.specialists(incident, parallel=False)
sequential_wall=time.perf_counter()-t
t=time.perf_counter()
parallel=await workshop.specialists(incident, parallel=True)
parallel_wall=time.perf_counter()-t
print('sequential:',round(sequential_wall,2),'parallel:',round(parallel_wall,2))
for role,out in parallel.items(): print('\n',role,'\n',out.text)

## 3. Synthesis and validation (30–40 min)
The LLM validator reads the original evidence; deterministic checks catch simple timeline inversions.

In [ ]:
swarm=await workshop.swarm(incident, verify=True)
print('SYNTHESIS\n',swarm['synthesis'].text)
print('VALIDATION\n',swarm['validation'].text)
print('RULE CHECK\n',swarm['deterministic_checks'])
print('CALLS:',swarm['requests'],'TOKENS:',swarm['input_tokens']+swarm['output_tokens'])

In [ ]:
bad_report='The severe weather began at 02:00 and definitively caused the 08:00 routing exception. The timeline is fully consistent.'
audit=await workshop.call('Audit the claim against the original incident. Flag false times and unsupported causality. Return PASS or FAIL with evidence.', 'ORIGINAL:\n'+json.dumps(incident)+'\nDRAFT:\n'+bad_report)
print(audit.text)
print(check_timeline(incident))

## 4. vLLM serving benchmark (40–50 min)
Compare requests/sec and latency for bounded concurrency. Execute on a warmed server, repeat runs for meaningful comparison.

In [ ]:
benchmarks=[]
for level in (1,2,4,8):
    result=await benchmark(workshop, incident, requests=20, concurrency=level)
    benchmarks.append(result)
    print(result)

In [ ]:
import matplotlib.pyplot as plt
plt.figure()
plt.plot([x['concurrency'] for x in benchmarks],[x['requests_per_second'] for x in benchmarks],marker='o')
plt.xlabel('Concurrent requests')
plt.ylabel('Requests / second')
plt.title('Observed vLLM throughput')
plt.show()

## 5. Closing discussion (50–60 min)
- Would the validator catch every unsupported claim?
- Is a larger agent swarm worth additional inference cost?
- What would change with distinct model instances?
- For production, use externally grounded checks, timezone-aware logs, human approvals, and repeated benchmark trials.

In [ ]:
await workshop.close()